# Multi-Architecture Trade Direction Prediction

## Approach
Use all market events (8 quote-update types + 4 trade types) as a sequence.
For each position in the sequence, predict the **volume-weighted buy-fraction**
within multiple forward event windows — without conditioning on the future.

Three model architectures are compared: **LSTM**, **Transformer**, **Mamba**.

### Why this approach?
- The all-events approach (Cell 24 of Transformer_Heating_Oil.ipynb) works — it
  meaningfully beats the last-trade-sign baseline on trade direction.
- The trade-level approach (Transformer_TradeLevel.py) does not work — aggregating
  30 quote updates into 9 summary features loses temporal signal.
- We fix the two problems with Cell 24:
  1. SEQ_LEN=100 captured only ~3-4 trades; we use SEQ_LEN=500 (~18 trades).
  2. Cell 24 predicted next-event type (conditions on future); we predict
     buy/sell fraction within a forward window (no future conditioning).

### Target: volume-weighted buy-fraction
For a forward window of H events after the history:
  buy_vol  = sum of trade sizes for Trade_B_* events in window
  sell_vol = sum of trade sizes for Trade_S_* events in window
  buy_frac = buy_vol / (buy_vol + sell_vol)
If no labeled trades in window, the sample is masked from the buy_frac loss.

### No-trade windows
An auxiliary binary head predicts P(has_trade in window).
The buy_frac loss is masked to only samples where has_trade=1.

In [ ]:
# =====================================================================
# SECTION 0: IMPORTS & CONFIGURATION
# =====================================================================
import datetime as dt
import math
import os
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# ---------------------
# 0a. File paths
# ---------------------
PARQUET_FILE = "HOH4_merged_quotes_trades.parquet"
LARGE_NEG    = -9223372036854775808   # INT64_MIN sentinel in raw data

# ---------------------
# 0b. Sequence / horizon
# ---------------------
SEQ_LEN  = 500              # events of history (~18 trades); try 1000 for richer context
HORIZONS = [50, 100, 200, 500]  # forward event windows to predict (~1.8, 3.6, 7.3, 18.2 trades)

# ---------------------
# 0c. Shared model dims
# ---------------------
D_MODEL = 128
DROPOUT = 0.25   # increased from 0.10 — signal is weak, model overfits quickly

# ---------------------
# 0d. LSTM config
# ---------------------
LSTM_HIDDEN = 128
LSTM_LAYERS = 2

# ---------------------
# 0e. Transformer config
# ---------------------
TF_NHEAD  = 4
TF_LAYERS = 4
TF_FFN    = 256

# ---------------------
# 0f. Mamba config
# ---------------------
MAMBA_D_STATE  = 8    # reduced from 16: each scan step is 2x cheaper
MAMBA_D_CONV   = 4
MAMBA_EXPAND   = 1    # reduced from 2: d_inner = d_model (no expansion) for pure-PyTorch speed
MAMBA_LAYERS   = 4

# ---------------------
# 0g. Training
# ---------------------
BATCH_SIZE    = 256
LEARNING_RATE = 5e-5   # reduced from 3e-4 — val loss exploded in ep2, LR was too aggressive
WEIGHT_DECAY  = 1e-2
EPOCHS        = 15
GRAD_CLIP     = 1.0
TRADE_LOSS_W  = 0.10   # weight of has_trade auxiliary loss

# ---------------------
# 0h. Data split
# ---------------------
TRAIN_FRAC = 0.80
VAL_FRAC   = 0.10

# Trade_N rows are dropped during preprocessing — no window banning needed

In [ ]:
# =====================================================================
# SECTION 1: GPU / REPRODUCIBILITY
# =====================================================================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

def _probe_cuda() -> torch.device:
    """
    Returns a working device. If CUDA is visible but its kernels are incompatible
    with the PyTorch build ('no kernel image' error), falls back to CPU.
    This happens when the PyTorch binary was compiled for a different GPU SM version
    than the physical GPU on this node (e.g. PyTorch built for sm_80 on an sm_75 GPU).
    Run this to diagnose:
        python -c "import torch; print(torch.cuda.get_device_capability()); print(torch.cuda.get_arch_list())"
    """
    if not torch.cuda.is_available():
        print(">>> CUDA not available — using CPU")
        return torch.device("cpu")
    try:
        # Minimal smoke test: allocate + run a basic op on GPU
        _t = torch.zeros(4, device="cuda")
        _ = _t + _t
        del _t
        name = torch.cuda.get_device_name(0)
        cap  = torch.cuda.get_device_capability(0)
        arches = torch.cuda.get_arch_list()
        print(f">>> GPU OK: {name}  (sm_{cap[0]}{cap[1]})  PyTorch arches: {arches}")
        return torch.device("cuda")
    except Exception as e:
        print(f">>> GPU kernel test failed: {e}")
        cap = torch.cuda.get_device_capability(0)
        arches = torch.cuda.get_arch_list() if hasattr(torch.cuda, "get_arch_list") else "unknown"
        print(f"    GPU SM: sm_{cap[0]}{cap[1]},  PyTorch built for: {arches}")
        print("    Falling back to CPU. To use this GPU, activate a conda env with a")
        print("    PyTorch build that includes sm_{}{} kernels.".format(cap[0], cap[1]))
        return torch.device("cpu")

DEVICE = _probe_cuda()

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

USE_AMP = (DEVICE.type == "cuda")
NUM_WORKERS = min(4, os.cpu_count() or 1) if DEVICE.type == "cuda" else 0
PIN_MEMORY  = (DEVICE.type == "cuda")

## Section 2: Preprocessing

In [ ]:
# =====================================================================
# SECTION 2a: preprocessing_time
#   - Merges date + time -> datetime
#   - Filters to market hours 07:00-13:30 CT, excludes weekends
#   - Labels ambiguous trades (aggr='N') as B/S using quote-change heuristic
# =====================================================================
def preprocessing_time(df: pd.DataFrame) -> pd.DataFrame:
    df_new = df.copy()
    df_new["datetime"] = df_new["date"] + df_new["time"]
    df_new.drop(columns=["date", "time"], inplace=True)
    df_new["datetime"] = pd.to_datetime(df_new["datetime"])

    mask = (
        ~df_new["datetime"].dt.date.isin([dt.date(2024, 2, 10), dt.date(2024, 2, 11)])
        & (df_new["datetime"].dt.hour >= 7)
        & (
            (df_new["datetime"].dt.hour < 13)
            | ((df_new["datetime"].dt.hour == 13) & (df_new["datetime"].dt.minute < 30))
        )
    )
    df_new = df_new[mask]

    is_N_trades = df_new["aggr"] == "N"
    print(f"Unlabelled trades: {is_N_trades.sum()}")

    same_dt_as_next = df_new["datetime"] == df_new["datetime"].shift(-1)
    ask_size_decrease = (df_new["asiz"].shift(1) > df_new["asiz"].shift(-1)) & (
        (df_new["asiz"].shift(1) - df_new["asiz"].shift(-1)) == df_new["siz"]
    )
    bid_size_decrease = (df_new["bsiz"].shift(1) > df_new["bsiz"].shift(-1)) & (
        (df_new["bsiz"].shift(1) - df_new["bsiz"].shift(-1)) == df_new["siz"]
    )
    ask_price_up  = df_new["ask"].shift(1) < df_new["ask"].shift(-1)
    bid_price_down = df_new["bid"].shift(1) > df_new["bid"].shift(-1)

    cond_B = is_N_trades & same_dt_as_next & (ask_size_decrease | ask_price_up)
    cond_S = is_N_trades & same_dt_as_next & (bid_size_decrease | bid_price_down)
    both   = cond_B & cond_S

    df_new.loc[both, "aggr"]          = "N"
    df_new.loc[cond_B & ~both, "aggr"] = "B"
    df_new.loc[cond_S & ~both, "aggr"] = "S"

    return df_new

In [ ]:
# =====================================================================
# SECTION 2b: preprocess_market_data
#   - Creates event stream: one row per quote change or trade
#   - Event types: 8 quote (Bid/Ask Px/Sz Up/Dn) + 4 trade (Trade_N dropped)
#   - Features: LogTime (log1p inter-event microseconds), LogSize (log1p size)
#   - Drops No_Change and Start rows
# =====================================================================
def preprocess_market_data(df: pd.DataFrame) -> pd.DataFrame:
    print(">>> Preprocessing market data...")

    sort_cols = ["datetime"] + (["seq"] if "seq" in df.columns else [])
    df = df.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)

    cols_to_clean = ["bsiz", "bid", "siz", "ask", "asiz"]
    cleaned = {}
    for c in cols_to_clean:
        cleaned[c] = df[c].where(df[c] != LARGE_NEG, np.nan)

    dt_us   = df["datetime"].diff().dt.total_seconds().fillna(0.0).to_numpy() * 1e6
    log_time = np.log1p(dt_us)

    n     = len(df)
    event = np.full(n, "No_Change", dtype=object)
    log_size = np.zeros(n, dtype=np.float64)

    if n > 0:
        event[0]    = "Start"
        log_size[0] = 0.0

    quote_cols = ["bid", "bsiz", "ask", "asiz"]
    q = pd.DataFrame({c: cleaned[c] for c in quote_cols}, index=df.index).ffill()

    bid  = q["bid"].to_numpy();  ask  = q["ask"].to_numpy()
    bsiz = q["bsiz"].to_numpy(); asiz = q["asiz"].to_numpy()

    bid_prev  = np.roll(bid,  1);  bid_prev[0]  = np.nan
    ask_prev  = np.roll(ask,  1);  ask_prev[0]  = np.nan
    bsiz_prev = np.roll(bsiz, 1);  bsiz_prev[0] = np.nan
    asiz_prev = np.roll(asiz, 1);  asiz_prev[0] = np.nan
    bid_next  = np.roll(bid, -1);  bid_next[-1] = np.nan
    ask_next  = np.roll(ask, -1);  ask_next[-1] = np.nan

    bsiz_delta = bsiz - bsiz_prev
    asiz_delta = asiz - asiz_prev

    siz        = cleaned["siz"].to_numpy()
    trade_mask = ~np.isnan(siz)
    side       = df["aggr"].fillna("N").astype(str).str.upper().to_numpy(dtype=str)
    valid      = np.arange(n) > 0

    conds = [
        valid & (bid > bid_prev),
        valid & (bid < bid_prev),
        valid & (bsiz_delta > 0),
        valid & (bsiz_delta < 0),
        valid & (ask > ask_prev),
        valid & (ask < ask_prev),
        valid & (asiz_delta > 0),
        valid & (asiz_delta < 0),
    ]
    choices = [
        "Bid_Px_Up", "Bid_Px_Dn", "Bid_Sz_Up", "Bid_Sz_Dn",
        "Ask_Px_Up", "Ask_Px_Dn", "Ask_Sz_Up", "Ask_Sz_Dn",
    ]
    quote_event    = np.select(conds, choices, default="No_Change")
    quote_log_size = np.select(
        conds,
        [1.0, 1.0, np.abs(bsiz_delta), np.abs(bsiz_delta), 1.0, 1.0,
         np.abs(asiz_delta), np.abs(asiz_delta)],
        default=0.0,
    )
    event[:] = quote_event
    log_size[:] = np.log1p(quote_log_size)

    trade_apply = valid & trade_mask
    base_trade  = np.char.add("Trade_", side).astype(object)

    is_B = trade_apply & (side == "B");  is_S = trade_apply & (side == "S")
    B_px_chg = is_B & (ask_next > ask_prev)
    S_px_chg = is_S & (bid_next < bid_prev)

    base_trade[B_px_chg]         = "Trade_B_PxChg"
    base_trade[is_B & ~B_px_chg] = "Trade_B_PxSame"
    base_trade[S_px_chg]         = "Trade_S_PxChg"
    base_trade[is_S & ~S_px_chg] = "Trade_S_PxSame"

    event[trade_apply]    = base_trade[trade_apply]
    log_size[trade_apply] = np.log1p(siz[trade_apply])

    if n > 0:
        event[0] = "Start"; log_size[0] = 0.0

    out = pd.DataFrame({"Event": event, "LogTime": log_time, "LogSize": log_size})
    # Drop non-informative rows: No_Change, Start, and Trade_N (ambiguous direction)
    out = out[~out["Event"].isin(["No_Change", "Start", "Trade_N"])].reset_index(drop=True)
    print(f"    Events after filtering: {len(out):,}")
    print(out["Event"].value_counts().to_string())
    return out

## Section 3: Dataset

In [ ]:
# =====================================================================
# SECTION 3: MultiHorizonDataset
#
# Each sample is a sliding window of SEQ_LEN events.
# Target: for each forward horizon H, compute volume-weighted buy_frac
#   buy_vol  = sum(raw_size for Trade_B_* in horizon)
#   sell_vol = sum(raw_size for Trade_S_* in horizon)
#   buy_frac = buy_vol / (buy_vol + sell_vol), masked when no trades
#
# Volume-weighting captures price impact better than simple trade counts.
# Prefix sums make all horizon computations O(1) per sample.
# =====================================================================
class MultiHorizonDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        seq_len: int,
        horizons: list,
        vocab: dict = None,
        feat_mean: np.ndarray = None,
        feat_std:  np.ndarray = None,
        buy_frac_prior: dict = None,    # {h: float} unconditional mean buy_frac
        cond_buy_frac:  dict = None,    # {h: {sign: float}} calibrated baseline
    ):
        self.seq_len     = seq_len
        self.horizons    = horizons
        self.max_horizon = max(horizons)

        events_str  = df["Event"].astype(str).tolist()
        log_time_np = df["LogTime"].to_numpy(dtype=np.float32)
        log_size_np = df["LogSize"].to_numpy(dtype=np.float32)
        n = len(df)

        # --- 3a. Vocabulary ---
        if vocab is None:
            unique_events = sorted(set(events_str))
            self.vocab = {ev: i for i, ev in enumerate(unique_events)}
            self.vocab["<UNK>"] = len(self.vocab)
        else:
            self.vocab = vocab

        unk_id    = self.vocab.get("<UNK>", 0)
        event_ids = np.array([self.vocab.get(e, unk_id) for e in events_str], dtype=np.int64)

        # --- 3b. Normalize continuous features ---
        cont_arr = np.stack([log_time_np, log_size_np], axis=1)  # [N, 2]
        if feat_mean is None:
            self.feat_mean = cont_arr.mean(axis=0)
            self.feat_std  = cont_arr.std(axis=0) + 1e-8
        else:
            self.feat_mean = feat_mean
            self.feat_std  = feat_std
        cont_norm = ((cont_arr - self.feat_mean) / self.feat_std).astype(np.float32)

        # --- 3c. Event type masks ---
        # Trade_N already removed in preprocessing — no need to filter windows
        buy_mask  = np.array([1.0 if e.startswith("Trade_B") else 0.0 for e in events_str])
        sell_mask = np.array([1.0 if e.startswith("Trade_S") else 0.0 for e in events_str])

        # Raw trade size: siz = exp(LogSize) - 1 (LogSize = log1p(siz) for trades)
        raw_size = np.expm1(log_size_np.astype(np.float64))

        # --- 3d. Prefix sums (prepend 0 for easy range queries) ---
        #   range [a, b) = prefix[b] - prefix[a]
        # Volume-weighted (primary target)
        prefix_buy_vol  = np.concatenate([[0.0], np.cumsum(buy_mask  * raw_size)])
        prefix_sell_vol = np.concatenate([[0.0], np.cumsum(sell_mask * raw_size)])
        # Count-based (diagnostic only — stored for comparison with volume-weighted)
        prefix_buy_cnt  = np.concatenate([[0.0], np.cumsum(buy_mask)])
        prefix_sell_cnt = np.concatenate([[0.0], np.cumsum(sell_mask)])
        self._prefix_buy_cnt  = prefix_buy_cnt   # exposed for diagnostic cell
        self._prefix_sell_cnt = prefix_sell_cnt

        # --- 3e. Valid start positions ---
        # Simple: need s + seq_len + max_horizon <= n
        if n < seq_len + self.max_horizon:
            self.valid_starts = np.array([], dtype=np.int64)
        else:
            s_max = n - seq_len - self.max_horizon
            self.valid_starts = np.arange(s_max + 1, dtype=np.int64)

        # --- 3f. Vectorized target computation ---
        t_arr = self.valid_starts + seq_len   # first event after history window
        n_h   = len(horizons)
        m     = len(self.valid_starts)

        buy_frac_tgts  = np.zeros((m, n_h), dtype=np.float32)
        has_trade_tgts = np.zeros((m, n_h), dtype=np.float32)

        for hi, h in enumerate(horizons):
            bv = prefix_buy_vol[t_arr + h]  - prefix_buy_vol[t_arr]
            sv = prefix_sell_vol[t_arr + h] - prefix_sell_vol[t_arr]
            tv = bv + sv
            has_trade_tgts[:, hi] = (tv > 0).astype(np.float32)
            buy_frac_tgts[:, hi]  = np.where(tv > 0, bv / tv, 0.5)  # 0.5 = masked placeholder

        self.buy_frac_tgts  = buy_frac_tgts
        self.has_trade_tgts = has_trade_tgts

        # --- 3g. Last-trade sign per event (vectorized forward fill) ---
        # last_sign_arr[i] = 1 (Buy), 0 (Sell), -1 (no prior trade)
        sign_raw = np.select(
            [buy_mask > 0, sell_mask > 0],
            [1.0, 0.0],
            default=np.nan,
        )
        last_sign_series = pd.Series(sign_raw).ffill().fillna(-1)
        self.last_sign_arr = last_sign_series.to_numpy(dtype=np.int8)

        # --- 3h. Baseline statistics (computed from train split, passed to val/test) ---
        if buy_frac_prior is None and m > 0:
            last_sign_per_sample = self.last_sign_arr[self.valid_starts + seq_len - 1]

            self.buy_frac_prior = {}
            self.cond_buy_frac  = {}

            for hi, h in enumerate(horizons):
                mask_has = has_trade_tgts[:, hi] > 0
                bf       = buy_frac_tgts[:, hi]

                prior = float(bf[mask_has].mean()) if mask_has.sum() > 0 else 0.5
                self.buy_frac_prior[h] = prior

                cond = {-1: prior}   # fallback when no prior trade in window
                for sign_val, sign_key in [(1, 1), (0, 0)]:
                    sign_mask = mask_has & (last_sign_per_sample == sign_val)
                    cond[sign_key] = float(bf[sign_mask].mean()) if sign_mask.sum() > 0 else prior
                self.cond_buy_frac[h] = cond
        else:
            self.buy_frac_prior = buy_frac_prior or {h: 0.5 for h in horizons}
            self.cond_buy_frac  = cond_buy_frac  or {h: {1: 0.5, 0: 0.5, -1: 0.5} for h in horizons}

        # Store arrays
        self.event_ids = event_ids
        self.cont_norm = cont_norm
        print(f"    Dataset size: {m:,} samples, no-trade rates: "
              + " | ".join(
                  f"H{h}={100*(1-has_trade_tgts[:,hi].mean()):.1f}%"
                  for hi, h in enumerate(horizons)
              ))

    def __len__(self):
        return len(self.valid_starts)

    def __getitem__(self, idx):
        s = int(self.valid_starts[idx])
        x_events   = torch.from_numpy(self.event_ids[s : s + self.seq_len].copy())
        x_cont     = torch.from_numpy(self.cont_norm[s : s + self.seq_len].copy())
        y_buy_frac  = torch.from_numpy(self.buy_frac_tgts[idx].copy())
        y_has_trade = torch.from_numpy(self.has_trade_tgts[idx].copy())
        last_sign   = int(self.last_sign_arr[s + self.seq_len - 1])
        return x_events, x_cont, y_buy_frac, y_has_trade, last_sign

## Section 4: Models

In [ ]:
# =====================================================================
# SECTION 4a: SHARED COMPONENTS
#   - SinusoidalPositionalEncoding: fixed sin/cos PE (no learned params)
#   - EventInputEncoder: shared input layer for all three architectures
#     embedding(event) + MLP(cont_feats) + PE + LayerNorm + Dropout
#   - PredictionHead: per-horizon MLP heads for buy_frac and has_trade
# =====================================================================
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int):
        super().__init__()
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[: pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)  # [1, L, D]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


class EventInputEncoder(nn.Module):
    """
    Shared input encoding for all three architectures.
    Combines event token embedding with continuous feature projection,
    adds sinusoidal positional encoding, then normalizes.
    """
    def __init__(self, num_event_types: int, d_model: int, max_seq_len: int, dropout: float):
        super().__init__()
        self.event_emb   = nn.Embedding(num_event_types, d_model)
        self.cont_proj   = nn.Sequential(
            nn.Linear(2, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.input_norm  = nn.LayerNorm(d_model)
        self.pos_enc     = SinusoidalPositionalEncoding(d_model, max_seq_len)
        self.dropout     = nn.Dropout(dropout)

    def forward(self, events: torch.Tensor, cont: torch.Tensor) -> torch.Tensor:
        """
        events: [B, S]   LongTensor of event token IDs
        cont:   [B, S, 2] normalized LogTime, LogSize
        Returns: [B, S, D]
        """
        x = self.event_emb(events) + self.cont_proj(cont)
        x = self.input_norm(x)
        x = self.pos_enc(x)
        return self.dropout(x)


class PredictionHead(nn.Module):
    """
    Per-horizon output heads shared by all architectures.
    Two heads per horizon:
      - buy_frac_head: predicts buy_frac logit (sigmoid -> [0,1])
      - has_trade_head: predicts P(has_trade) logit
    """
    def __init__(self, d_model: int, n_horizons: int, dropout: float):
        super().__init__()
        mid = max(d_model // 2, 32)
        # buy_frac heads: deeper to capture direction nuance
        self.buy_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, mid),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(mid, 1),
            )
            for _ in range(n_horizons)
        ])
        # has_trade heads: simpler (more structural, less directional)
        self.trade_heads = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, mid // 2), nn.GELU(), nn.Linear(mid // 2, 1))
            for _ in range(n_horizons)
        ])

    def forward(self, h: torch.Tensor):
        """
        h: [B, D] last hidden state
        Returns:
          buy_logits   [B, n_horizons]
          trade_logits [B, n_horizons]
        """
        buy_logits   = torch.cat([head(h) for head in self.buy_heads],   dim=-1)
        trade_logits = torch.cat([head(h) for head in self.trade_heads], dim=-1)
        return buy_logits, trade_logits

In [ ]:
# =====================================================================
# SECTION 4b: MARKET LSTM
#   Unidirectional LSTM (causal by construction, no mask needed).
#   EventInputEncoder -> LSTM -> last timestep -> PredictionHead
# =====================================================================
class MarketLSTM(nn.Module):
    """
    Causal LSTM for multi-horizon trade direction prediction.
    Unidirectional to avoid using future information.
    """
    def __init__(
        self,
        num_event_types: int,
        d_model: int,
        hidden_dim: int,
        n_layers: int,
        dropout: float,
        n_horizons: int,
        seq_len: int,
    ):
        super().__init__()
        self.encoder = EventInputEncoder(num_event_types, d_model, seq_len, dropout)
        self.lstm    = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        # Project LSTM hidden dim to d_model if they differ
        self.proj = (
            nn.Linear(hidden_dim, d_model) if hidden_dim != d_model else nn.Identity()
        )
        self.head = PredictionHead(d_model, n_horizons, dropout)

    def forward(self, events: torch.Tensor, cont: torch.Tensor):
        x = self.encoder(events, cont)          # [B, S, D]
        out, _ = self.lstm(x)                   # [B, S, hidden]
        last = self.proj(out[:, -1, :])         # [B, D]
        return self.head(last)

In [ ]:
# =====================================================================
# SECTION 4c: MARKET TRANSFORMER
#   Decoder-only (causal self-attention via upper-triangular mask).
#   Same proven architecture as Cell 24 of Transformer_Heating_Oil.ipynb,
#   but with multi-horizon PredictionHead instead of next-event-type head.
# =====================================================================
class MarketTransformer(nn.Module):
    """
    GPT-style decoder-only transformer for multi-horizon trade prediction.
    Uses causal (upper-triangular) mask to prevent future information leakage.
    """
    def __init__(
        self,
        num_event_types: int,
        d_model: int,
        nhead: int,
        n_layers: int,
        ffn_dim: int,
        dropout: float,
        n_horizons: int,
        seq_len: int,
    ):
        super().__init__()
        self.encoder = EventInputEncoder(num_event_types, d_model, seq_len, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,   # pre-norm (more stable)
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers,
            norm=nn.LayerNorm(d_model),
        )
        self.head = PredictionHead(d_model, n_horizons, dropout)

        # Causal mask: upper triangle is -inf (position i cannot attend to j > i)
        self.register_buffer(
            "causal_mask",
            torch.triu(torch.full((seq_len, seq_len), float("-inf")), diagonal=1),
            persistent=False,
        )

    def forward(self, events: torch.Tensor, cont: torch.Tensor):
        x = self.encoder(events, cont)                          # [B, S, D]
        seq = x.size(1)
        h = self.transformer(x, mask=self.causal_mask[:seq, :seq])  # [B, S, D]
        last = h[:, -1, :]                                      # [B, D]
        return self.head(last)

In [ ]:
# =====================================================================
# SECTION 4d: MARKET MAMBA
#   State-space model (SSM) architecture. Naturally causal (no mask needed).
#   Efficient for long sequences: O(L) vs O(L^2) for Transformer.
#   Particularly advantageous for SEQ_LEN=1000.
#
#   Implementation:
#   - Uses mamba_ssm package if installed (fast CUDA kernels)
#   - Falls back to pure-PyTorch sequential scan otherwise
#     (correct but slower; ~10x slower on GPU, use for debugging only)
# =====================================================================
try:
    from mamba_ssm import Mamba as _MambaKernel
    _HAS_MAMBA_SSM = True
    print(">>> mamba_ssm available: using fast CUDA kernels for Mamba")
except ImportError:
    _HAS_MAMBA_SSM = False
    print(">>> mamba_ssm not installed. Using pure-PyTorch sequential scan.")


def _selective_scan_sequential(
    x: torch.Tensor,       # [B, L, d_inner]
    delta: torch.Tensor,   # [B, L, d_inner]
    A: torch.Tensor,       # [d_inner, d_state]  (already negated)
    B_mat: torch.Tensor,   # [B, L, d_state]
    C_mat: torch.Tensor,   # [B, L, d_state]
    D: torch.Tensor,       # [d_inner]
) -> torch.Tensor:         # [B, L, d_inner]
    """
    Sequential SSM scan — computes dA_t and dB_t per step to avoid the
    O(B*L*d_inner*d_state) precomputed tensor (would be ~2 GB for default dims).
    Peak memory: O(B * d_inner * d_state) for the hidden state h.
    For each t:
        dA_t = exp(delta_t * A)       [B, d_inner, d_state]
        dB_t = delta_t * B_t          [B, d_inner, d_state]
        h_t  = dA_t * h_{t-1} + dB_t * x_t
        y_t  = C_t @ h_t + D * x_t
    """
    B_sz, L, d_inner = x.shape
    d_state = A.shape[1]
    h = x.new_zeros(B_sz, d_inner, d_state)
    ys = []
    for t in range(L):
        dt = delta[:, t]                              # [B, d_inner]
        dA_t = torch.exp(dt.unsqueeze(-1) * A)       # [B, d_inner, d_state]
        dB_t = dt.unsqueeze(-1) * B_mat[:, t].unsqueeze(1)  # [B, d_inner, d_state]
        h = dA_t * h + dB_t * x[:, t].unsqueeze(-1) # [B, d_inner, d_state]
        y_t = (h * C_mat[:, t].unsqueeze(1)).sum(-1) + D * x[:, t]  # [B, d_inner]
        ys.append(y_t)
    return torch.stack(ys, dim=1)  # [B, L, d_inner]


class _MambaBlockPyTorch(nn.Module):
    """
    Pure-PyTorch Mamba SSM block.
    Architecture: in_proj -> causal conv1d -> SiLU -> SSM -> gate -> out_proj
    Residual + pre-norm wrapping is done in MarketMamba.
    """
    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.d_inner = d_model * expand
        self.d_state = d_state

        # Gate projection: x -> (x_main, z) each of size d_inner
        self.in_proj  = nn.Linear(d_model, self.d_inner * 2, bias=False)

        # Causal depthwise conv (padding = d_conv-1, trimmed after to keep causal)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner,
            kernel_size=d_conv,
            padding=d_conv - 1,
            groups=self.d_inner,
            bias=True,
        )

        # Input-dependent SSM projections: delta (1), B (d_state), C (d_state)
        self.x_proj  = nn.Linear(self.d_inner, 1 + d_state + d_state, bias=False)
        self.dt_proj = nn.Linear(1, self.d_inner, bias=True)

        # A: diagonal SSM matrix, log-space [d_inner, d_state]
        A_init = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(self.d_inner, -1)
        self.A_log = nn.Parameter(torch.log(A_init))

        # Skip connection
        self.D = nn.Parameter(torch.ones(self.d_inner))

        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B, L, d_model] -> [B, L, d_model]"""
        B, L, _ = x.shape

        # Gate split
        xz = self.in_proj(x)                        # [B, L, d_inner*2]
        x_main, z = xz.chunk(2, dim=-1)             # each [B, L, d_inner]

        # Causal depthwise conv
        x_conv = self.conv1d(x_main.transpose(1, 2))[:, :, :L].transpose(1, 2)  # [B, L, d_inner]
        x_conv = F.silu(x_conv)

        # Input-dependent parameters
        params   = self.x_proj(x_conv)              # [B, L, 1 + d_state + d_state]
        delta_1  = params[:, :, :1]                 # [B, L, 1]
        B_mat    = params[:, :, 1:1 + self.d_state] # [B, L, d_state]
        C_mat    = params[:, :, 1 + self.d_state:]  # [B, L, d_state]

        delta = F.softplus(self.dt_proj(delta_1))   # [B, L, d_inner]

        # A matrix (negated log-space → negative real values for stability)
        A = -torch.exp(self.A_log)   # [d_inner, d_state]

        # Sequential scan — dA/dB computed per-step inside to avoid O(B*L*d*s) tensor
        y = _selective_scan_sequential(x_conv, delta, A, B_mat, C_mat, self.D)  # [B, L, d_inner]

        # Gating: y * SiLU(z)
        y = y * F.silu(z)

        return self.out_proj(y)                     # [B, L, d_model]


class _MambaBlockFast(nn.Module):
    """Thin wrapper around mamba_ssm.Mamba (fast CUDA kernels)."""
    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.mamba = _MambaKernel(
            d_model=d_model,
            d_state=d_state,
            d_conv=d_conv,
            expand=expand,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mamba(x)


_MambaBlockCls = _MambaBlockFast if _HAS_MAMBA_SSM else _MambaBlockPyTorch


class MarketMamba(nn.Module):
    """
    Mamba SSM stack for multi-horizon trade direction prediction.
    Naturally causal (no attention mask needed), O(L) in sequence length.
    Best architecture for SEQ_LEN=1000 where Transformer attention is expensive.
    """
    def __init__(
        self,
        num_event_types: int,
        d_model: int,
        d_state: int,
        d_conv: int,
        expand: int,
        n_layers: int,
        dropout: float,
        n_horizons: int,
        seq_len: int,
    ):
        super().__init__()
        self.encoder = EventInputEncoder(num_event_types, d_model, seq_len, dropout)

        self.blocks = nn.ModuleList([
            _MambaBlockCls(d_model, d_state, d_conv, expand)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.drop  = nn.Dropout(dropout)
        self.head  = PredictionHead(d_model, n_horizons, dropout)

    def forward(self, events: torch.Tensor, cont: torch.Tensor):
        x = self.encoder(events, cont)   # [B, S, D]
        for block, norm in zip(self.blocks, self.norms):
            x = x + self.drop(block(norm(x)))   # pre-norm residual
        last = x[:, -1, :]               # [B, D]
        return self.head(last)

## Section 5: Evaluation & Baselines

In [ ]:
# =====================================================================
# SECTION 5: EVALUATION
#
# Three baselines per horizon (all calibrated — no hard 0/1 probabilities):
#   1. Last-Trade Calibrated: P(buy_frac | last_sign, h) from training data
#      (primary baseline — mirrors last-trade-sign autocorrelation)
#   2. Window Volume Prior: buy_frac within the history window
#   3. Unconditional Prior: training-set mean buy_frac
#
# All metrics restricted to samples where has_trade=1 (trade-containing horizons).
#
# Metrics:
#   accuracy    : (pred > 0.5) == (true > 0.5)
#   log_loss    : BCE loss with clipped probabilities
#   brier       : mean squared error of probability vs true
#   info_gain   : baseline_logloss - model_logloss (positive = model wins)
# =====================================================================
_EPS = 1e-7


def _bce_loss_np(p: np.ndarray, y: np.ndarray) -> float:
    p = np.clip(p, _EPS, 1 - _EPS)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


@torch.no_grad()
def evaluate(
    model,
    loader: DataLoader,
    dataset: MultiHorizonDataset,
    horizons: list,
    eps: float = _EPS,
) -> dict:
    """
    Returns summary dict:
      {h: {model_logloss, model_acc, model_brier,
           base_lastcal_logloss, base_lastcal_acc, base_lastcal_brier, info_gain_lastcal,
           base_window_logloss, base_window_info_gain,
           base_prior_logloss,  base_prior_info_gain,
           n_eval}}
    """
    model.eval()
    n_h = len(horizons)

    # Accumulators per horizon: model preds, true buy_frac, has_trade, last_sign
    all_preds    = [[] for _ in range(n_h)]
    all_true     = [[] for _ in range(n_h)]
    all_has      = [[] for _ in range(n_h)]
    all_last_sgn = []

    # For window-prior baseline, we need buy vol from history window
    all_win_prior = [[] for _ in range(n_h)]   # per-sample window buy_frac

    for batch in loader:
        x_ev, x_ct, y_bf, y_ht, last_sign = batch
        x_ev = x_ev.to(DEVICE)
        x_ct = x_ct.to(DEVICE)

        buy_logits, _ = model(x_ev, x_ct)   # [B, n_h]
        preds = torch.sigmoid(buy_logits).cpu().numpy()   # [B, n_h]

        y_bf_np = y_bf.numpy()
        y_ht_np = y_ht.numpy()
        ls_np   = last_sign.numpy()

        # Window buy_frac: fraction of buy vol among all trade vol in history window
        # Use event_ids in x_ev to identify Trade_B / Trade_S tokens
        x_ev_cpu = x_ev.cpu().numpy()  # [B, S]
        buy_ids  = {idx for ev, idx in dataset.vocab.items() if isinstance(ev, str) and ev.startswith("Trade_B")}
        sell_ids = {idx for ev, idx in dataset.vocab.items() if isinstance(ev, str) and ev.startswith("Trade_S")}

        # Precompute LogSize -> raw size from x_ct
        # x_ct is normalized; we don't have raw logsize in the batch
        # Use a flat window prior (buy_frac_prior) as a simpler window baseline instead
        # since recovering raw size from normalized features requires feat_mean/std
        x_ct_cpu = x_ct.cpu().numpy()

        for b in range(preds.shape[0]):
            row_ev = x_ev_cpu[b]
            # Count buy and sell events (size-1 proxy since raw size not easily available here)
            b_count = sum(1 for eid in row_ev if eid in buy_ids)
            s_count = sum(1 for eid in row_ev if eid in sell_ids)
            t_count = b_count + s_count
            win_bf  = (b_count / t_count) if t_count > 0 else dataset.buy_frac_prior[horizons[0]]
            for hi in range(n_h):
                all_win_prior[hi].append(win_bf)

        for hi in range(n_h):
            all_preds[hi].append(preds[:, hi])
            all_true[hi].append(y_bf_np[:, hi])
            all_has[hi].append(y_ht_np[:, hi])
        all_last_sgn.append(ls_np)

    last_sgn_all = np.concatenate(all_last_sgn)

    summary = {}
    for hi, h in enumerate(horizons):
        preds_h = np.concatenate(all_preds[hi])
        true_h  = np.concatenate(all_true[hi])
        has_h   = np.concatenate(all_has[hi]).astype(bool)
        win_h   = np.array(all_win_prior[hi])

        # Restrict to trade-containing samples
        p = preds_h[has_h];   y = true_h[has_h]
        ls = last_sgn_all[has_h];  wv = win_h[has_h]
        n_eval = int(has_h.sum())

        if n_eval == 0:
            summary[h] = {"n_eval": 0}
            continue

        # Last-trade calibrated baseline
        cond = dataset.cond_buy_frac[h]
        base_lc = np.array([cond[int(s)] for s in ls])

        # Unconditional prior
        base_pr = np.full(n_eval, dataset.buy_frac_prior[h])

        model_ll     = _bce_loss_np(p, y)
        base_lc_ll   = _bce_loss_np(base_lc, y)
        base_win_ll  = _bce_loss_np(np.clip(wv, eps, 1-eps), y)
        base_pr_ll   = _bce_loss_np(base_pr, y)

        summary[h] = {
            "n_eval":              n_eval,
            "model_logloss":       model_ll,
            "model_acc":           float(((p > 0.5) == (y > 0.5)).mean()),
            "model_brier":         float(((p - y) ** 2).mean()),
            "base_lastcal_logloss":  base_lc_ll,
            "base_lastcal_acc":      float(((base_lc > 0.5) == (y > 0.5)).mean()),
            "base_lastcal_brier":    float(((base_lc - y) ** 2).mean()),
            "info_gain_lastcal":     base_lc_ll - model_ll,
            "base_window_logloss":   base_win_ll,
            "info_gain_window":      base_win_ll - model_ll,
            "base_prior_logloss":    base_pr_ll,
            "info_gain_prior":       base_pr_ll - model_ll,
        }

    return summary


def print_results_table(summary: dict, horizons: list, model_name: str = "Model"):
    """Pretty-print per-horizon results for a single model."""
    h_strs = [f"H={h:>4}" for h in horizons]
    header = f"\n{'─'*70}\n  {model_name} — Test Set Results\n{'─'*70}"
    print(header)
    print(f"{'Metric':<28}" + "".join(f"{s:>12}" for s in h_strs))
    print("─" * 70)

    rows = [
        ("n_eval",              "Samples (has_trade)"),
        ("model_logloss",       "Model log-loss"),
        ("base_lastcal_logloss","Baseline log-loss (last-trade cal.)"),
        ("info_gain_lastcal",   "Info gain vs last-trade"),
        ("model_acc",           "Model accuracy"),
        ("base_lastcal_acc",    "Baseline accuracy"),
        ("model_brier",         "Model Brier score"),
        ("info_gain_window",    "Info gain vs window prior"),
        ("info_gain_prior",     "Info gain vs uncond. prior"),
    ]
    for key, label in rows:
        vals = []
        for h in horizons:
            v = summary.get(h, {}).get(key, float("nan"))
            if key == "n_eval":
                vals.append(f"{int(v):>12,}")
            else:
                vals.append(f"{v:>+12.4f}" if isinstance(v, float) else f"{v:>12}")
        print(f"{label:<28}" + "".join(vals))
    print("─" * 70)


def print_cross_model_comparison(results: dict, horizons: list):
    """Side-by-side comparison of all models for each horizon."""
    models = list(results.keys())
    print(f"\n{'═'*80}\n  CROSS-MODEL COMPARISON\n{'═'*80}")
    for h in horizons:
        print(f"\n  Horizon H={h} events")
        print(f"  {'Metric':<28}" + "".join(f"{m:>12}" for m in models))
        print("  " + "─" * (28 + 12 * len(models)))
        for key, label in [
            ("model_logloss",      "Log-loss"),
            ("base_lastcal_logloss","Baseline log-loss"),
            ("info_gain_lastcal",  "Info gain (last-trade)"),
            ("model_acc",          "Accuracy"),
            ("model_brier",        "Brier"),
        ]:
            vals = []
            for m in models:
                v = results[m][1].get(h, {}).get(key, float("nan"))
                vals.append(f"{v:>+12.4f}" if isinstance(v, float) else f"{v:>12}")
            print(f"  {label:<28}" + "".join(vals))

## Section 6: Training Loop

In [ ]:
# =====================================================================
# SECTION 6: TRAINING LOOP
#
# Loss:
#   For each horizon h independently:
#     buy_frac_loss[h] = BCEWithLogitsLoss on has_trade=1 samples only
#     has_trade_loss[h] = BCEWithLogitsLoss on all samples
#   total = mean(buy_frac_loss) + TRADE_LOSS_W * mean(has_trade_loss)
#
# Optimizer: AdamW with cosine LR schedule
# AMP: mixed precision on CUDA
# Early stopping: restore best val weights
# =====================================================================
_bce_logits = nn.BCEWithLogitsLoss()

def _compute_loss(
    buy_logits:   torch.Tensor,   # [B, n_h]
    trade_logits: torch.Tensor,   # [B, n_h]
    y_buy_frac:   torch.Tensor,   # [B, n_h]
    y_has_trade:  torch.Tensor,   # [B, n_h]
) -> torch.Tensor:
    n_h = buy_logits.size(1)
    buy_loss   = torch.tensor(0.0, device=buy_logits.device)
    trade_loss = torch.tensor(0.0, device=buy_logits.device)

    for hi in range(n_h):
        mask = y_has_trade[:, hi] > 0.5
        if mask.sum() > 0:
            buy_loss = buy_loss + _bce_logits(buy_logits[mask, hi], y_buy_frac[mask, hi])
        trade_loss = trade_loss + _bce_logits(trade_logits[:, hi], y_has_trade[:, hi])

    return buy_loss / n_h + TRADE_LOSS_W * trade_loss / n_h


def train_single_model(
    model_class,
    model_kwargs: dict,
    train_ds: MultiHorizonDataset,
    val_ds:   MultiHorizonDataset,
    test_ds:  MultiHorizonDataset,
    horizons: list,
    model_name: str,
    epochs: int = EPOCHS,
    lr: float = LEARNING_RATE,
    wd: float = WEIGHT_DECAY,
):
    print(f"\n{'━'*70}")
    print(f"  Training: {model_name}")
    print(f"{'━'*70}")

    model = model_class(**model_kwargs).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parameters: {n_params:,}")

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd, betas=(0.9, 0.95))
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler    = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best_val_ll = float("inf")
    best_state  = None

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        train_loss = 0.0
        n_batches  = 0
        for x_ev, x_ct, y_bf, y_ht, _ in train_loader:
            x_ev = x_ev.to(DEVICE); x_ct = x_ct.to(DEVICE)
            y_bf = y_bf.to(DEVICE); y_ht = y_ht.to(DEVICE)

            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                buy_logits, trade_logits = model(x_ev, x_ct)
                loss = _compute_loss(buy_logits, trade_logits, y_bf, y_ht)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            n_batches  += 1

        train_loss /= max(n_batches, 1)

        # --- Validate ---
        val_summary = evaluate(model, val_loader, val_ds, horizons)
        val_ll = np.mean([
            val_summary[h]["model_logloss"]
            for h in horizons
            if val_summary.get(h, {}).get("n_eval", 0) > 0
        ])

        # Early stopping
        if val_ll < best_val_ll:
            best_val_ll = val_ll
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step()

        # Per-epoch log
        gains = " | ".join(
            f"H{h}:{val_summary.get(h,{}).get('info_gain_lastcal', float('nan')):+.4f}"
            for h in horizons
        )
        print(
            f"  Ep {epoch:02d}/{epochs}  "
            f"TrLoss={train_loss:.4f}  "
            f"ValLL={val_ll:.4f}  "
            f"InfoGain({gains})"
        )

    # Restore best weights and evaluate on test set
    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    test_summary = evaluate(model, test_loader, test_ds, horizons)
    print_results_table(test_summary, horizons, model_name)
    return model, test_summary

## Section 7: Run All Models

In [ ]:
# =====================================================================
# SECTION 7: RUN ALL MODELS
#   1. Load and preprocess data
#   2. Temporal split (80/10/10 by event index)
#   3. Build datasets with shared vocab, normalization, baseline stats
#   4. Train LSTM, Transformer, Mamba sequentially
#   5. Print cross-model comparison
# =====================================================================
def run_all():
    # --- 7a. Load data ---
    print(f">>> Loading {PARQUET_FILE}")
    raw_df = pd.read_parquet(PARQUET_FILE)
    print(f"    Raw rows: {len(raw_df):,}")

    clean_df    = preprocessing_time(raw_df)
    processed_df = preprocess_market_data(clean_df)
    n_total     = len(processed_df)

    # --- 7b. Temporal split ---
    t1 = int(TRAIN_FRAC * n_total)
    t2 = int((TRAIN_FRAC + VAL_FRAC) * n_total)
    train_df = processed_df.iloc[:t1].reset_index(drop=True)
    val_df   = processed_df.iloc[t1:t2].reset_index(drop=True)
    test_df  = processed_df.iloc[t2:].reset_index(drop=True)
    print(f"\n>>> Split: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,} events")

    # --- 7c. Build datasets ---
    print("\n>>> Building datasets...")
    print("  Train:")
    train_ds = MultiHorizonDataset(train_df, SEQ_LEN, HORIZONS)

    print("  Val:")
    val_ds = MultiHorizonDataset(
        val_df, SEQ_LEN, HORIZONS,
        vocab=train_ds.vocab,
        feat_mean=train_ds.feat_mean,
        feat_std=train_ds.feat_std,
        buy_frac_prior=train_ds.buy_frac_prior,
        cond_buy_frac=train_ds.cond_buy_frac,
    )

    print("  Test:")
    test_ds = MultiHorizonDataset(
        test_df, SEQ_LEN, HORIZONS,
        vocab=train_ds.vocab,
        feat_mean=train_ds.feat_mean,
        feat_std=train_ds.feat_std,
        buy_frac_prior=train_ds.buy_frac_prior,
        cond_buy_frac=train_ds.cond_buy_frac,
    )

    # --- 7d. Baseline diagnostic ---
    # Verifies that cond_buy_frac is computed only on has_trade=1 windows,
    # and checks whether volume-weighting explains any gap vs count-based autocorrelation.
    print("\n>>> Baseline calibration diagnostic (train):")
    last_sign_arr = train_ds.last_sign_arr[train_ds.valid_starts + SEQ_LEN - 1]
    for hi, h in enumerate(HORIZONS):
        bf  = train_ds.buy_frac_tgts[:, hi]
        ht  = train_ds.has_trade_tgts[:, hi] > 0
        stored = train_ds.cond_buy_frac[h]

        # Direct recompute — should exactly match stored values
        buy_m  = ht & (last_sign_arr == 1)
        sell_m = ht & (last_sign_arr == 0)
        direct_buy  = float(bf[buy_m].mean())  if buy_m.sum()  > 0 else float("nan")
        direct_sell = float(bf[sell_m].mean()) if sell_m.sum() > 0 else float("nan")

        # Count-based buy_frac (not volume-weighted) — for comparison
        # Each trade event counts as 1 regardless of size.
        # Uses stored _prefix_buy_cnt / _prefix_sell_cnt arrays.
        t_arr   = train_ds.valid_starts + SEQ_LEN
        bv_cnt  = train_ds._prefix_buy_cnt[t_arr + h]  - train_ds._prefix_buy_cnt[t_arr]
        sv_cnt  = train_ds._prefix_sell_cnt[t_arr + h] - train_ds._prefix_sell_cnt[t_arr]
        tv_cnt  = bv_cnt + sv_cnt
        cnt_ht  = tv_cnt > 0
        cnt_bf  = np.where(cnt_ht, bv_cnt / np.where(cnt_ht, tv_cnt, 1.0), 0.5)
        cnt_prior     = float(cnt_bf[cnt_ht].mean()) if cnt_ht.sum() > 0 else float("nan")
        cnt_buy_m     = cnt_ht & (last_sign_arr == 1)
        cnt_sell_m    = cnt_ht & (last_sign_arr == 0)
        cnt_cond_buy  = float(cnt_bf[cnt_buy_m].mean())  if cnt_buy_m.sum()  > 0 else float("nan")
        cnt_cond_sell = float(cnt_bf[cnt_sell_m].mean()) if cnt_sell_m.sum() > 0 else float("nan")

        print(f"  H={h:>4}: no-trade={100*(~ht).mean():.1f}%  n_buy={buy_m.sum():,}  n_sell={sell_m.sum():,}")
        print(f"    vol-weighted (model target):  prior={train_ds.buy_frac_prior[h]:.4f}"
              f"  cond(Buy)={stored[1]:.4f}  cond(Sell)={stored[0]:.4f}"
              f"  [verify: direct_buy={direct_buy:.4f}  direct_sell={direct_sell:.4f}]")
        print(f"    count-based   (diagnostic):  prior={cnt_prior:.4f}"
              f"  cond(Buy)={cnt_cond_buy:.4f}  cond(Sell)={cnt_cond_sell:.4f}"
              f"  <- higher cond(Buy) expected if vol-weighting dilutes autocorrelation")

    num_events = len(train_ds.vocab)
    print(f"\n>>> Vocab size: {num_events} (including <UNK>)")

    # --- 7e. Common model kwargs ---
    common = dict(
        num_event_types=num_events,
        d_model=D_MODEL,
        dropout=DROPOUT,
        n_horizons=len(HORIZONS),
        seq_len=SEQ_LEN,
    )

    model_configs = [
        (
            "LSTM",
            MarketLSTM,
            {**common, "hidden_dim": LSTM_HIDDEN, "n_layers": LSTM_LAYERS},
        ),
        (
            "Transformer",
            MarketTransformer,
            {**common, "nhead": TF_NHEAD, "n_layers": TF_LAYERS, "ffn_dim": TF_FFN},
        ),
        (
            "Mamba",
            MarketMamba,
            {**common, "d_state": MAMBA_D_STATE, "d_conv": MAMBA_D_CONV,
             "expand": MAMBA_EXPAND, "n_layers": MAMBA_LAYERS},
        ),
    ]

    # --- 7f. Train all models ---
    results = {}
    for name, cls, kwargs in model_configs:
        model, test_summary = train_single_model(
            cls, kwargs, train_ds, val_ds, test_ds, HORIZONS, name
        )
        results[name] = (model, test_summary)

    # --- 7g. Cross-model comparison ---
    all_summaries = {name: (m, s) for name, (m, s) in results.items()}
    print_cross_model_comparison(all_summaries, HORIZONS)

    return results

In [ ]:
if __name__ == "__main__":
    results = run_all()